# Import libraries

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import itertools
import math
import time

# Function

In [ ]:
# Function to prepare df (merge questions with responses)
def prepare_dataframe(question_df_path, model_a_df_path, model_b_df_path=None):
    """
    Parameters:
    - question_df_path: Path to the questions file (transformed_questions.csv)
    - model_a_df_path: Path to the response file of model A
    - model_b_df_path: Path to the response file of model B (optional, if None then single)

    Returns:
    - Dataframe merged with columns as shown in the figure: question_id, category, content_turn1, response_turn1_a,
      content_turn2, response_turn2_a, response_turn1_b, response_turn2_b, reference_turn1, reference_turn2
    """
    # Read question_df
    df = pd.read_csv(question_df_path)

    # Tách turn1 và turn2
    turn1_df = df[df['turn'] == 1].drop(columns=['turn'])
    turn2_df = df[df['turn'] == 2].drop(columns=['turn'])

    turn1_df = turn1_df.rename(columns={
        'Vietnamese_content': 'content_turn1',
        'Vietnamese_reference': 'reference_turn1'
    })
    turn2_df = turn2_df.rename(columns={
        'Vietnamese_content': 'content_turn2',
        'Vietnamese_reference': 'reference_turn2'
    })

    # Merge turn1 and turn2
    merged_df = pd.merge(turn1_df, turn2_df, on=['question_id', 'category'], how='left')

    # Read model A response
    model_a_df = pd.read_csv(model_a_df_path)
    model_a_df = model_a_df.rename(columns={
        'response_turn1': 'response_turn1_a',
        'response_turn2': 'response_turn2_a'
    })

    # Merge with model A
    merged_df = pd.merge(merged_df, model_a_df[['question_id', 'category', 'response_turn1_a', 'response_turn2_a']],
                         on=['question_id', 'category'], how='left')

    # If model B is provided
    if model_b_df_path:
        model_b_df = pd.read_csv(model_b_df_path)
        model_b_df = model_b_df.rename(columns={
            'response_turn1': 'response_turn1_b',
            'response_turn2': 'response_turn2_b'
        })
        merged_df = pd.merge(merged_df, model_b_df[['question_id', 'category', 'response_turn1_b', 'response_turn2_b']],
                             on=['question_id', 'category'], how='left')
    else:
        # If no model B is provided, add NaN columns
        merged_df['response_turn1_b'] = pd.NA
        merged_df['response_turn2_b'] = pd.NA

    merged_df = merged_df[['question_id', 'category', 'content_turn1', 'reference_turn1', 'response_turn1_a', 'response_turn1_b',
                           'content_turn2', 'reference_turn2', 'response_turn2_a', 'response_turn2_b',  ]]

    return merged_df

# Merge

In [ ]:
# Adjust file path to match the target environment
question_path = 'drive/MyDrive/ViJudge/data/questions/questions_2_turns.csv'

In [ ]:
'''
ID for mapping LLMs
ID,name_of_LLMs_responses_file
01,openai_gpt-oss-20b_response.csv
02,llama-3.3-70b-versatile_response.csv
03,moonshotai_kimi-k2-instruct_response.csv
04,qwen_qwen3-32b_response.csv
05,deepseek_deepseek-chat-v3.1:free_response.csv
'''

'''
Method of mapping id
Single: 1abcdef
Pair: 2abcdef
In which:
    - ab: original question id
    - cd: id of model A
    - ef: id of model B (if any)
'''

dict_llms = {
    1: ['openai', 'openai_gpt-oss-20b_response.csv'],
    2: ['llama', 'llama-3.3-70b-versatile_response.csv'],
    3: ['moonshotai', 'moonshotai_kimi-k2-instruct_response.csv'],
    4: ['qwen', 'qwen_qwen3-32b_response.csv'],
    5: ['deepseek', 'deepseek_deepseek-chat-v3.1:free_response.csv']
}

id_list = list(dict_llms.keys())
combinations = []
# Single: create sets [model_a, nan]
for combo in itertools.combinations(id_list, 1):
    combinations.append([combo[0], math.nan])
# Pair: create sets [model_a, model_b]
for combo in itertools.combinations(id_list, 2):
    combinations.append(list(combo))

process = 0
for item in combinations:
    id_model_a = item[0]
    id_model_b = item[1]
    print(f"Model_a: {dict_llms[id_model_a][0]}")
    model_a_response_path = 'drive/MyDrive/ViJudge/data/LLMs_responses/' + dict_llms[id_model_a][1]
    # Single
    if math.isnan(id_model_b):
        print(f"Model_b: {math.nan}")
        merged_df = prepare_dataframe(question_path, model_a_response_path)
        merged_df['model_a'] = dict_llms[id_model_a][0]
        merged_df['model_b'] = pd.NA
        merged_df['question_id'] = 1000000 + merged_df['question_id']*10000 + id_model_a*100
        output_file_path = 'drive/MyDrive/ViJudge/data/datasets_before_judgement/single_' + dict_llms[id_model_a][0] + '.csv'
    # Pair
    else:
        print(f"Model_b: {dict_llms[id_model_b][0]}")
        model_b_response_path = 'drive/MyDrive/ViJudge/data/LLMs_responses/' + dict_llms[id_model_b][1]
        merged_df = prepare_dataframe(question_path, model_a_response_path, model_b_response_path)
        merged_df['model_a'] = dict_llms[id_model_a][0]
        merged_df['model_b'] = dict_llms[id_model_b][0]
        merged_df['question_id'] = 2000000 + merged_df['question_id']*10000 + id_model_a*100 + id_model_b
        output_file_path = 'drive/MyDrive/ViJudge/data/datasets_before_judgement/pair_' + dict_llms[id_model_a][0] + '_' + dict_llms[id_model_b][0] + '.csv'

    merged_df = merged_df[['question_id', 'category', 'model_a', 'model_b', 'content_turn1', 'reference_turn1', 'response_turn1_a', 'response_turn1_b',
                           'content_turn2', 'reference_turn2', 'response_turn2_a', 'response_turn2_b',  ]]
    merged_df.to_csv(output_file_path, index=False)
    print(f"Written to file '{output_file_path}'")
    process += 1
    print(f"Done: {process}/{len(combinations)}\n")
    time.sleep(2)